# PanganFlow.ID - Experiment Notebook

Notebook ini mendokumentasikan eksperimen inti PanganFlow.ID untuk GEMASTIK Penambangan Data: konstruksi panel harga-neraca beras, evaluasi ranking provinsi prioritas, clustering, dan rekomendasi flow redistribusi antarprovinsi.

Scope v1: **Beras Medium** pada level provinsi. Output adalah prioritas tinjauan kebijakan, bukan keputusan distribusi final.

## 0. Setup

Jalankan notebook dari root folder `PanganFlow-ID`. Kalau notebook dibuka dari folder `notebooks`, cell setup di bawah tetap mencoba menemukan root proyek secara otomatis.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src' / 'build_dataset.py').exists() else cwd.parent
DATA = ROOT / 'data' / 'processed'
TABLES = ROOT / 'reports' / 'tables'
FIGURES = ROOT / 'reports' / 'figures'

print('ROOT =', ROOT)
print('DATA exists =', DATA.exists())
print('TABLES exists =', TABLES.exists())

## 1. Rebuild Artifacts Jika Dibutuhkan

Cell ini tidak otomatis menjalankan pipeline setiap kali notebook dibuka. Ubah `RUN_PIPELINE = True` kalau ingin rebuild dataset, model, dan laporan dari awal.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    for cmd in [
        [sys.executable, 'src/build_dataset.py'],
        [sys.executable, 'src/train_panganflow.py'],
        [sys.executable, 'src/build_formal_report.py'],
    ]:
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print('Pipeline rebuild skipped. Set RUN_PIPELINE = True to regenerate artifacts.')

## 2. Load Dataset dan Tabel Eksperimen

In [ ]:
panel = pd.read_csv(DATA / 'province_commodity_panel.csv', parse_dates=['date'])
flows = pd.read_csv(DATA / 'flow_recommendations.csv')
metrics = pd.read_csv(TABLES / 'metrics_summary.csv')
ranking = pd.read_csv(TABLES / 'ranking_snapshot.csv')
importance = pd.read_csv(TABLES / 'feature_importance.csv')
ablation = pd.read_csv(TABLES / 'feature_ablation.csv')
clusters = pd.read_csv(TABLES / 'cluster_snapshot.csv')
sources = pd.read_csv(TABLES / 'source_status.csv')
validation = pd.read_csv(TABLES / 'validation_checks.csv')

print('panel rows:', len(panel))
print('province count:', panel['province'].nunique())
print('period:', panel['date'].min().date(), 'to', panel['date'].max().date())
print('flow candidates:', len(flows))

## 3. Dataset Summary

Panel utama menggabungkan harga beras, sinyal perubahan harga, gap terhadap median nasional, produksi tahunan, estimasi kebutuhan, dan proxy surplus-defisit.

In [ ]:
summary = {
    'rows': len(panel),
    'provinces': panel['province'].nunique(),
    'commodities': panel['commodity'].nunique(),
    'date_min': panel['date'].min().strftime('%Y-%m-%d'),
    'date_max': panel['date'].max().strftime('%Y-%m-%d'),
    'duplicate_province_period_rows': int(panel.duplicated(['province', 'date', 'commodity']).sum()),
}
pd.Series(summary).to_frame('value')

In [ ]:
sources[['source', 'status', 'rows', 'notes']].head(10)

## 4. Harga Beras dan Disparitas Nasional

Sinyal harga tidak dipakai sendiri. PanganFlow.ID menambahkan konteks median nasional dan proxy neraca pangan supaya prioritas tidak hanya berarti harga tertinggi.

In [ ]:
latest_date = panel['date'].max()
latest_panel = panel[panel['date'].eq(latest_date)].copy()
latest_panel[['province', 'price', 'national_median_gap', 'surplus_proxy_ton', 'deficit_score', 'surplus_score', 'anomaly_score']].sort_values('price', ascending=False).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_price = latest_panel.sort_values('price', ascending=False).head(12).iloc[::-1]
ax.barh(top_price['province'], top_price['price'], color='#145c72')
ax.axvline(latest_panel['national_median_price'].median(), color='#b5442d', linestyle='--', label='median nasional')
ax.set_title(f'Top Harga Beras Medium - {latest_date:%Y-%m}')
ax.set_xlabel('Rp/kg')
ax.legend()
ax.grid(axis='x', alpha=.2)
plt.tight_layout()

## 5. Evaluasi Ranking

Model utama adalah hybrid: weighted priority index yang transparan dikalibrasi dengan Random Forest. Ini menjaga interpretabilitas sekaligus memberi ruang untuk interaksi non-linear.

In [ ]:
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
m = metrics.sort_values('ndcg_at_k')
ax.barh(m['model'], m['ndcg_at_k'], color='#0b6b5f')
ax.set_title('Perbandingan NDCG@10')
ax.set_xlabel('NDCG@10')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()

In [ ]:
ranking.head(12)

## 6. Feature Importance dan Ablation

Bagian ini membantu menjawab apakah ranking hanya didorong harga, atau benar-benar memakai kombinasi harga, neraca, dan fitur spasial.

In [ ]:
importance.head(15)

In [ ]:
ablation

## 7. Rekomendasi Flow Surplus -> Defisit

Flow score menggabungkan selisih harga, skor defisit tujuan, skor surplus asal, skor model tujuan, anomali tujuan, dan penalti jarak. Output ini adalah antrian review, bukan instruksi distribusi otomatis.

In [ ]:
flows[['rank', 'origin_province', 'destination_province', 'priority_score', 'distance_km', 'top_reasons']].head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_flows = flows.head(10).iloc[::-1]
labels = top_flows['origin_province'] + ' -> ' + top_flows['destination_province']
ax.barh(labels, top_flows['priority_score'], color='#b5442d')
ax.set_title('Top-10 Rekomendasi Flow Beras')
ax.set_xlabel('Priority Score')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()

## 8. Clustering Provinsi

Clustering dipakai untuk memberi tipologi wilayah: surplus relatif stabil, defisit-harga tinggi, rawan shock, atau transisi/moderat.

In [ ]:
clusters[['province', 'cluster_label', 'model_priority_score', 'deficit_score', 'surplus_score', 'anomaly_score']].sort_values('model_priority_score', ascending=False).head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, group in clusters.groupby('cluster_label'):
    ax.scatter(group['pca_x'], group['pca_y'], label=label, s=70, alpha=.85)
for _, row in clusters.sort_values('model_priority_score', ascending=False).head(6).iterrows():
    ax.text(row['pca_x'] + .05, row['pca_y'] + .05, row['province'], fontsize=8)
ax.set_title('PCA Cluster Provinsi')
ax.set_xlabel('PCA-1')
ax.set_ylabel('PCA-2')
ax.legend(fontsize=8)
ax.grid(alpha=.2)
plt.tight_layout()

## 9. Validasi dan Checklist

Validasi utama: join provinsi, duplicate row, leakage rolling window, flow no self-pair, origin surplus, ranking stability, dan citation hygiene.

In [ ]:
validation

In [ ]:
assert panel.duplicated(['province', 'date', 'commodity']).sum() == 0
assert flows['origin_province'].ne(flows['destination_province']).all()
assert flows['priority_score'].between(0, 100).all()
assert panel['province'].nunique() >= 34
print('Notebook validation checks passed.')

## 10. Reproducibility Commands

```powershell
cd "D:\\Home\\Projects\\GEMASTIK\\DATA MINING - Penambangan Data\\PanganFlow-ID"
uv run python src/build_dataset.py
uv run python src/train_panganflow.py
uv run python src/build_formal_report.py
uv run streamlit run dashboard/app.py
```

Catatan: file `data/raw/bps_rice_balance_snapshot.csv` adalah seed snapshot manual resmi/proxy. Untuk klaim final, ganti dengan export BPS yang diunduh tim lalu rerun pipeline.